%md
# 02_transform_silver_alerts — Capa Silver de señales TradingView

## 1. Objetivo de la capa Silver

La capa **Silver** transforma los eventos crudos ingeridos en Bronze desde S3 en una tabla limpia, estructurada, deduplicada y lista para consumo analítico.

La fuente principal es:

    trading.bronze.alerts_raw

El destino principal es:

    trading.silver.alerts_clean

---

## 2. Funcionalidades principales

La capa Silver realiza las siguientes funciones:

1. Lee eventos desde la tabla Bronze `trading.bronze.alerts_raw`.
2. Extrae campos principales del evento.
3. Aplana campos anidados de `raw_payload`.
4. Aplana campos anidados de `raw_validation`.
5. Conserva `raw_payload` y `raw_validation` completos para trazabilidad.
6. Normaliza campos como `symbol` y `side`.
7. Extrae información de validación, trade plan, scoring y probabilidad.
8. Extrae `market_snapshot` calculado por el backend.
9. Extrae `structure_snapshot`.
10. Extrae contexto reutilizado de la alerta.
11. Extrae campos de ML tracking.
12. Genera una clave única técnica para deduplicación.
13. Ejecuta un `MERGE` incremental hacia Delta.
14. Evita duplicados por `event_unique_key`.
15. Deja la tabla preparada para capas Gold y datasets ML.

---

## 3. Rol dentro del Lakehouse

Flujo actual:

    TradingView
      ↓
    Vercel /api/webhook
      ↓
    AWS SQS
      ↓
    AWS Lambda Consumer
      ↓
    Supabase operacional
      ↓
    AWS S3 Bronze JSONL
      ↓
    Databricks Auto Loader
      ↓
    trading.bronze.alerts_raw
      ↓
    02_transform_silver_alerts
      ↓
    trading.silver.alerts_clean

La capa Silver es el punto donde los datos dejan de ser principalmente raw/semi-estructurados y pasan a tener columnas explícitas para análisis, ML, backtesting y futuras capas Gold.

---

## 4. Campos principales aplanados

Silver expone directamente campos operativos básicos:

    event_uid
    schema_version
    message_type
    trace_id
    symbol
    symbol_normalized
    tf
    event
    side
    side_normalized
    payload_price

Estos campos permiten consultas rápidas sin navegar estructuras JSON anidadas.

---

## 5. Campos de validación

Desde `raw_validation.validation`, Silver extrae:

    validation_approve
    validation_confidence
    probability_tp_before_sl
    entry_price
    tp
    sl
    rr
    score_external
    quality_score_alert
    probability_model
    barrier_component
    technical_component
    ml_component

Estos campos permiten analizar si una señal fue aprobada o rechazada y con qué nivel de confianza, probabilidad y score externo.

---

## 6. Market snapshot

Silver aplana el bloque `market_snapshot` calculado por el backend:

    close_1m
    ema20_1m
    ema50_1m
    ema200_1m
    adx_1m
    plus_di_1m
    minus_di_1m
    atr14_1m
    rvol20_1m
    impulse_atr_1m
    dist_to_vwap_pct_1m
    spread_bps
    book_imbalance
    buy_aggression
    sell_aggression
    delta_qty
    bid_wall_detected
    ask_wall_detected
    vacuum_above
    vacuum_below

Este bloque permite analizar técnicamente la señal usando datos de mercado de Binance en el momento de la validación.

---

## 7. Structure snapshot

Silver extrae el bloque de estructura de mercado:

    last_swing_high
    last_swing_low
    distance_to_swing_high_pct
    distance_to_swing_low_pct
    range_mode
    compression_box

Estos campos ayudan a evaluar si había espacio suficiente hacia TP o si la señal estaba limitada por estructura cercana.

---

## 8. Contexto reutilizado de la alerta

Desde `raw_validation.validation.alert_reused`, Silver extrae:

    regime
    phase
    dir_state
    mov_state
    liq_state
    htf_phase
    htf_phase_strength
    trigger_alignment
    too_extended_warn_alert
    too_extended_block_alert
    late_trend_alert

Esto permite cruzar la lectura del backend con el contexto original enviado por TradingView.

---

## 9. Campos para ML tracking

Desde `raw_payload.ml_tracking`, Silver extrae:

    track_for_outcome
    candidate_type
    labeling_profile
    ml_target
    ambiguous_rule
    timeout_rule
    entry_reference
    tp_sl_source

Estos campos son la base para construir datasets futuros de Machine Learning y etiquetado de outcomes.

Ejemplo conceptual:

    labeling_window_bars = 20
    ml_target = tp_before_sl
    candidate_type = INIT
    entry_reference = payload_entry_price_else_close
    tp_sl_source = payload_trade_plan

---

## 10. Trazabilidad

Aunque Silver aplana campos importantes, conserva:

    raw_payload
    raw_validation

Esto permite:

1. Auditar cualquier decisión.
2. Reprocesar lógica futura sin volver a S3.
3. Comparar versiones de esquema.
4. Extraer nuevos campos si el Alert Engine evoluciona.
5. Construir nuevas tablas Gold sin perder contexto.

---

## 11. Deduplicación

Silver genera una clave técnica:

    event_unique_key

Construida con:

    event_uid || message_type || event

Esto evita duplicados cuando:

1. Lambda reintenta un mensaje.
2. SQS entrega más de una vez.
3. Auto Loader procesa archivos en varias ejecuciones.
4. El notebook Silver se ejecuta repetidamente.

---

## 12. Escritura incremental

El notebook usa:

    foreachBatch + Delta MERGE

La lógica es:

    Si event_unique_key ya existe:
        actualizar registro

    Si event_unique_key no existe:
        insertar registro

Esto permite ejecuciones incrementales, reprocesos controlados e idempotencia en la tabla Silver.

---

## 13. Modo de ejecución

El notebook usa:

    trigger(availableNow=True)

Esto significa:

    Procesa todos los datos disponibles en Bronze
    ↓
    Escribe en Silver
    ↓
    Se detiene automáticamente

Este modo es adecuado para ejecuciones manuales, jobs programados, backfills controlados y procesamiento batch incremental.

---

## 14. Estado actual

VALIDADO:

- Lectura desde Bronze.
- Transformación hacia Silver.
- Aplanamiento de campos principales.
- Extracción de validación.
- Extracción de `market_snapshot`.
- Extracción de `structure_snapshot`.
- Conservación de `raw_payload` / `raw_validation`.
- Escritura Delta en `trading.silver.alerts_clean`.
- Deduplicación con `event_unique_key`.
- Ejecución incremental con checkpoint.

PENDIENTE:

- Crear capa Gold.
- Crear tabla de candidatos ML.
- Crear lógica de outcome labeling.
- Cruzar señales con precios posteriores para saber si TP o SL se alcanzó primero.
- Construir métricas agregadas por setup, fase, régimen, dirección y calidad.

---

## 15. Resumen ejecutivo

La capa `02_transform_silver_alerts` convierte los eventos Bronze crudos en una tabla Silver limpia, deduplicada y analíticamente útil.

La tabla resultante:

    trading.silver.alerts_clean

queda preparada como fuente para:

- ML.
- Backtesting.
- Dashboards.
- Métricas de estrategia.
- Auditoría de señales.
- Tablas Gold.

In [0]:
# ============================================================
# REBUILD SILVER ALERTS CLEAN FROM BRONZE - BATCH MODE
# Usar una vez para regenerar schema con signal_time
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

SOURCE_TABLE = "trading.bronze.alerts_raw"
TARGET_TABLE = "trading.silver.alerts_clean"
CHECKPOINT_PATH = "s3://trading-lakehouse-btc-s3/checkpoints/silver/alerts_clean/"

spark.sql("CREATE SCHEMA IF NOT EXISTS trading.silver")

# ------------------------------------------------------------
# Limpiar tabla Silver y checkpoint anterior
# ------------------------------------------------------------

spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

try:
    dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
    print(f"Checkpoint removed: {CHECKPOINT_PATH}")
except Exception as e:
    print(f"Checkpoint remove skipped or failed: {str(e)}")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def column_exists(df, path: str) -> bool:
    parts = path.split(".")
    current_schema = df.schema

    for part in parts:
        found = None

        for field in current_schema.fields:
            if field.name == part:
                found = field
                break

        if found is None:
            return False

        if isinstance(found.dataType, T.StructType):
            current_schema = found.dataType
        elif part != parts[-1]:
            return False

    return True


def safe_double(df, path: str, alias: str, default=None):
    if column_exists(df, path):
        return F.col(path).cast("double").alias(alias)
    return F.lit(default).cast("double").alias(alias)


def safe_boolean(df, path: str, alias: str, default=None):
    if column_exists(df, path):
        return F.col(path).cast("boolean").alias(alias)
    return F.lit(default).cast("boolean").alias(alias)


def safe_string(df, path: str, alias: str, default=None):
    if column_exists(df, path):
        return F.col(path).cast("string").alias(alias)
    return F.lit(default).cast("string").alias(alias)

def safe_timestamp(df, path: str, alias: str):
    if not column_exists(df, path):
        return F.lit(None).cast("timestamp").alias(alias)

    value_col = F.col(path)
    value_str = value_col.cast("string")

    return (
        F.when(
            value_str.rlike(r"^\d{13}$"),
            F.to_timestamp(F.from_unixtime(value_str.cast("double") / F.lit(1000.0)))
        )
        .when(
            value_str.rlike(r"^\d{10}$"),
            F.to_timestamp(F.from_unixtime(value_str.cast("double")))
        )
        .otherwise(
            F.try_to_timestamp(value_str)
        )
        .alias(alias)
    )
# ------------------------------------------------------------
# Read Bronze en batch, no streaming
# ------------------------------------------------------------

df_bronze = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# Crear Silver completo con signal_time
# ------------------------------------------------------------

df_silver = (
    df_bronze
    .select(
        F.col("event_uid").cast("string").alias("event_uid"),
        F.col("schema_version").cast("string").alias("schema_version"),
        F.col("message_type").cast("string").alias("message_type"),
        F.col("trace_id").cast("string").alias("trace_id"),

        F.col("symbol").cast("string").alias("symbol"),
        F.lower(F.col("symbol").cast("string")).alias("symbol_normalized"),
        F.col("tf").cast("string").alias("tf"),
        F.col("event").cast("string").alias("event"),
        F.col("side").cast("string").alias("side"),
        F.lower(F.col("side").cast("string")).alias("side_normalized"),

        safe_timestamp(df_bronze, "raw_payload.signal.bar_time", "signal_time"),
        safe_string(df_bronze, "raw_payload.signal.bar_time", "signal_bar_time_raw"),
        safe_string(df_bronze, "raw_payload.signal.timestamp", "signal_timestamp_raw"),

        F.col("price").cast("double").alias("payload_price"),

        F.col("validation_approve").cast("boolean").alias("validation_approve"),
        F.col("validation_confidence").cast("double").alias("validation_confidence"),
        F.col("probability_tp_before_sl").cast("double").alias("probability_tp_before_sl"),

        safe_double(df_bronze, "raw_validation.validation.entry_price", "entry_price"),
        safe_double(df_bronze, "raw_validation.validation.tp", "tp"),
        safe_double(df_bronze, "raw_validation.validation.sl", "sl"),
        safe_double(df_bronze, "raw_validation.validation.rr", "rr"),
        safe_double(df_bronze, "raw_validation.validation.score_external", "score_external"),
        safe_double(df_bronze, "raw_validation.validation.quality_score_alert", "quality_score_alert"),

        safe_string(df_bronze, "raw_validation.validation.probability_model", "probability_model"),
        safe_double(df_bronze, "raw_validation.validation.barrier_component", "barrier_component"),
        safe_double(df_bronze, "raw_validation.validation.technical_component", "technical_component"),
        safe_double(df_bronze, "raw_validation.validation.ml_component", "ml_component"),

        safe_double(df_bronze, "raw_validation.validation.market_snapshot.close_1m", "close_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.ema20_1m", "ema20_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.ema50_1m", "ema50_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.ema200_1m", "ema200_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.adx_1m", "adx_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.plus_di_1m", "plus_di_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.minus_di_1m", "minus_di_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.atr14_1m", "atr14_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.rvol20_1m", "rvol20_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.impulse_atr_1m", "impulse_atr_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.dist_to_vwap_pct_1m", "dist_to_vwap_pct_1m"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.spread_bps", "spread_bps"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.book_imbalance", "book_imbalance"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.buy_aggression", "buy_aggression"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.sell_aggression", "sell_aggression"),
        safe_double(df_bronze, "raw_validation.validation.market_snapshot.delta_qty", "delta_qty"),

        safe_boolean(df_bronze, "raw_validation.validation.market_snapshot.bid_wall_detected", "bid_wall_detected"),
        safe_boolean(df_bronze, "raw_validation.validation.market_snapshot.ask_wall_detected", "ask_wall_detected"),
        safe_boolean(df_bronze, "raw_validation.validation.market_snapshot.vacuum_above", "vacuum_above"),
        safe_boolean(df_bronze, "raw_validation.validation.market_snapshot.vacuum_below", "vacuum_below"),

        safe_double(df_bronze, "raw_validation.validation.structure_snapshot.last_swing_high", "last_swing_high"),
        safe_double(df_bronze, "raw_validation.validation.structure_snapshot.last_swing_low", "last_swing_low"),
        safe_double(df_bronze, "raw_validation.validation.structure_snapshot.distance_to_swing_high_pct", "distance_to_swing_high_pct"),
        safe_double(df_bronze, "raw_validation.validation.structure_snapshot.distance_to_swing_low_pct", "distance_to_swing_low_pct"),
        safe_boolean(df_bronze, "raw_validation.validation.structure_snapshot.range_mode", "range_mode"),
        safe_boolean(df_bronze, "raw_validation.validation.structure_snapshot.compression_box", "compression_box"),

        safe_string(df_bronze, "raw_validation.validation.alert_reused.regime", "regime"),
        safe_string(df_bronze, "raw_validation.validation.alert_reused.phase", "phase"),
        safe_string(df_bronze, "raw_validation.validation.alert_reused.dir_state", "dir_state"),
        safe_string(df_bronze, "raw_validation.validation.alert_reused.mov_state", "mov_state"),
        safe_string(df_bronze, "raw_validation.validation.alert_reused.liq_state", "liq_state"),
        safe_string(df_bronze, "raw_validation.validation.alert_reused.htf_phase", "htf_phase"),
        safe_double(df_bronze, "raw_validation.validation.alert_reused.htf_phase_strength", "htf_phase_strength"),
        safe_double(df_bronze, "raw_validation.validation.alert_reused.trigger_alignment", "trigger_alignment"),
        safe_boolean(df_bronze, "raw_validation.validation.alert_reused.too_extended_warn_alert", "too_extended_warn_alert"),
        safe_boolean(df_bronze, "raw_validation.validation.alert_reused.too_extended_block_alert", "too_extended_block_alert"),
        safe_boolean(df_bronze, "raw_validation.validation.alert_reused.late_trend_alert", "late_trend_alert"),

        safe_boolean(df_bronze, "raw_payload.ml_tracking.track_for_outcome", "track_for_outcome"),
        safe_string(df_bronze, "raw_payload.ml_tracking.candidate_type", "candidate_type"),
        safe_string(df_bronze, "raw_payload.ml_tracking.labeling_profile", "labeling_profile"),
        safe_string(df_bronze, "raw_payload.ml_tracking.ml_target", "ml_target"),
        safe_string(df_bronze, "raw_payload.ml_tracking.ambiguous_rule", "ambiguous_rule"),
        safe_string(df_bronze, "raw_payload.ml_tracking.timeout_rule", "timeout_rule"),
        safe_string(df_bronze, "raw_payload.ml_tracking.entry_reference", "entry_reference"),
        safe_string(df_bronze, "raw_payload.ml_tracking.tp_sl_source", "tp_sl_source"),

        F.col("raw_payload").alias("raw_payload"),
        F.col("raw_validation").alias("raw_validation"),

        F.col("ingestion_ts").cast("timestamp").alias("backend_ingestion_ts"),
        F.col("processing_date").cast("date").alias("processing_date"),
        F.col("lakehouse_layer").cast("string").alias("source_lakehouse_layer"),

        F.col("_databricks_ingestion_ts").cast("timestamp").alias("_databricks_ingestion_ts"),
        F.col("_source_file").cast("string").alias("_source_file"),
        F.col("_source_file_modification_time").cast("timestamp").alias("_source_file_modification_time"),
        F.current_timestamp().alias("_silver_processed_ts")
    )
    .withColumn(
        "event_unique_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("event_uid"), F.lit("")),
                F.coalesce(F.col("message_type"), F.lit("")),
                F.coalesce(F.col("event"), F.lit(""))
            ),
            256
        )
    )
    .withColumn(
        "is_validated_event",
        F.col("validation_approve").isNotNull()
    )
    .withColumn(
        "is_approved",
        F.coalesce(F.col("validation_approve"), F.lit(False))
    )
    .filter(F.col("event_uid").isNotNull())
    .filter(F.col("message_type").isNotNull())
)

# ------------------------------------------------------------
# Deduplicación determinística Silver
# ------------------------------------------------------------
# Motivo:
# - TradingView puede mostrar timeout y reenviar / duplicar eventos.
# - S3 Bronze puede contener más de un JSONL con el mismo event_uid.
# - Silver debe conservar 1 sola fila por evento lógico.
#
# Criterio:
# - event_unique_key = event_uid + message_type + event
# - conservar la versión más reciente según:
#   1. _source_file_modification_time
#   2. backend_ingestion_ts
#   3. _databricks_ingestion_ts
#   4. _source_file
#
# Nota:
# - dropDuplicates() no es determinístico.
# - row_number() permite elegir explícitamente qué duplicado conservar.

dedupe_window = (
    Window
    .partitionBy("event_unique_key")
    .orderBy(
        F.col("_source_file_modification_time").desc_nulls_last(),
        F.col("backend_ingestion_ts").desc_nulls_last(),
        F.col("_databricks_ingestion_ts").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last()
    )
)

df_silver = (
    df_silver
    .withColumn("_dedupe_rank", F.row_number().over(dedupe_window))
    .withColumn("_duplicate_count", F.count("*").over(Window.partitionBy("event_unique_key")))
    .withColumn("_is_duplicate_in_bronze", F.col("_duplicate_count") > 1)
    .filter(F.col("_dedupe_rank") == 1)
    .drop("_dedupe_rank")
)
# ------------------------------------------------------------
# Write Silver overwrite
# ------------------------------------------------------------

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Rebuilt table: {TARGET_TABLE}")
df_silver.select(
    "event_uid",
    "signal_time",
    "signal_bar_time_raw",
    "signal_timestamp_raw"
).where(
    F.col("message_type") == "logical_event_full"
).show(50, truncate=False)

# ------------------------------------------------------------
# Validación de duplicados Silver
# ------------------------------------------------------------

print("Duplicados por event_unique_key en Silver:")
(
    spark.table(TARGET_TABLE)
    .groupBy("event_unique_key")
    .count()
    .where(F.col("count") > 1)
    .show(50, truncate=False)
)

print("Eventos que venían duplicados desde Bronze pero fueron deduplicados en Silver:")
(
    spark.table(TARGET_TABLE)
    .where(F.col("_is_duplicate_in_bronze") == True)
    .select(
        "event_uid",
        "message_type",
        "event",
        "side",
        "_duplicate_count",
        "_source_file",
        "_source_file_modification_time",
        "backend_ingestion_ts"
    )
    .show(50, truncate=False)
)

print("Conteo total Silver:")
print(spark.table(TARGET_TABLE).count())
# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Schema contains signal_time?")
print("signal_time" in spark.table(TARGET_TABLE).columns)

spark.table(TARGET_TABLE).select(
    "event_uid",
    "message_type",
    "signal_time",
    "signal_bar_time_raw",
    "signal_timestamp_raw",
    "symbol",
    "tf",
    "event",
    "side",
    "entry_price",
    "tp",
    "sl"
).where(
    F.col("message_type") == "logical_event_full"
).show(50, truncate=False)


#### VALIDACION

In [0]:
%sql
-- ULTIMO ELEMENTO VALIDADO
select *
from trading.silver.alerts_clean
where event_uid = 'test_s3_final_010';

-- CONTEO
select
  message_type,
  event,
  validation_approve,
  count(*) as total
from trading.silver.alerts_clean
group by message_type, event, validation_approve
order by total desc;
